# Tractor Beam Comprehensive Tutorial

This notebook demonstrates both the GUI and programmatic interfaces for tractor-beam, providing a complete overview of capabilities.

## 1. Installation and Setup

First, let's make sure tractor-beam is properly installed and import the necessary components.

In [ ]:
# Import core tractor-beam components
from tractor_beam import tractor
from tractor_beam.utils.config import Config
import json
import asyncio
from pathlib import Path

print("Tractor Beam components imported successfully!")

## 2. GUI Interface

The GUI provides a user-friendly interface for all tractor-beam operations. Here's how to launch it:

In [ ]:
# Launch the GUI (uncomment to run)
# Note: This will open a separate window

# from tractor_beam.gui import TractorBeamGUI
# app = TractorBeamGUI()
# app.run()

print("To launch the GUI, uncomment the code above or run:")
print("python examples/gui_demo.py")

## 3. Creating Configurations

Let's create different types of configurations for various use cases.

In [ ]:
# Simple configuration for basic web scraping
simple_config = {
    "role": "server",
    "settings": {
        "name": "tutorial_basic",
        "proj_dir": "./tutorial_output",
        "jobs": [
            {
                "url": "https://httpbin.org/html",
                "types": ["html"],
                "beacon": None,
                "delay": None,
                "tasks": ["abduct", "visits"],
                "description": "Basic HTML scraping example"
            }
        ]
    }
}

print("Simple configuration created:")
print(json.dumps(simple_config, indent=2))

In [ ]:
# Advanced configuration with multiple jobs and processing
advanced_config = {
    "role": "server",
    "settings": {
        "name": "tutorial_advanced",
        "proj_dir": "./tutorial_advanced_output",
        "jobs": [
            {
                "url": "https://httpbin.org/html",
                "types": ["html"],
                "beacon": None,
                "delay": None,
                "tasks": ["abduct", "visits", "process"],
                "description": "HTML with text processing"
            },
            {
                "url": "https://httpbin.org/json",
                "types": ["json"],
                "beacon": None,
                "delay": None,
                "tasks": ["abduct", "visits"],
                "description": "JSON data collection",
                "custom": [
                    {
                        "func": "json_processor",
                        "headers": {
                            "User-Agent": "TractorBeam-Tutorial/1.0"
                        }
                    }
                ]
            }
        ]
    }
}

print("Advanced configuration created with multiple jobs and custom processing")

In [ ]:
# Monitoring configuration for periodic checking
monitoring_config = {
    "role": "watcher",
    "settings": {
        "name": "tutorial_monitor",
        "proj_dir": "./tutorial_monitor_output",
        "jobs": [
            {
                "url": "https://httpbin.org/delay/1",
                "types": ["json"],
                "beacon": None,
                "delay": 10,  # Check every 10 seconds (short for demo)
                "tasks": ["abduct", "visits"],
                "description": "Periodic monitoring example"
            }
        ]
    }
}

print("Monitoring configuration created for periodic data collection")

## 4. Configuration Validation

Let's validate our configurations to ensure they're properly formatted.

In [ ]:
# Validate configurations
configs_to_test = [
    ("Simple", simple_config),
    ("Advanced", advanced_config),
    ("Monitoring", monitoring_config)
]

for name, config in configs_to_test:
    try:
        # Create a Config object to validate
        config_obj = Config(config)
        print(f"✅ {name} configuration is valid")
    except Exception as e:
        print(f"❌ {name} configuration is invalid: {e}")

## 5. Programmatic Usage

Now let's use tractor-beam programmatically to execute our configurations.

In [ ]:
# Execute a simple configuration
async def run_simple_example():
    print("Starting simple tractor-beam execution...")
    
    try:
        # Create beam instance
        beam = tractor.Beam(simple_config)
        
        # Define callback for progress tracking
        async def progress_callback(runs):
            print(f"Completed {len(runs)} job runs")
            if runs:
                last_run = runs[-1]
                print(f"Last run status: {last_run.get('status', 'unknown')}")
        
        # Execute the beam
        results = await beam.go(cb=progress_callback)
        
        print("Simple execution completed successfully!")
        return results
        
    except Exception as e:
        print(f"Error during execution: {e}")
        return None

# Run the example (uncomment to execute)
# results = await run_simple_example()
print("Simple execution function defined. Uncomment the line above to run.")

In [ ]:
# Example with detailed progress tracking
async def run_with_detailed_tracking():
    print("Starting execution with detailed progress tracking...")
    
    beam = tractor.Beam(advanced_config)
    
    # Detailed progress callback
    async def detailed_callback(runs):
        print(f"\n--- Progress Update ---")
        print(f"Total completed runs: {len(runs)}")
        
        for i, run in enumerate(runs):
            print(f"Run {i+1}:")
            print(f"  Status: {run.get('status', 'unknown')}")
            
            # Show file counts if available
            if 'Abduct' in run and hasattr(run['Abduct'], 'state'):
                file_count = len(run['Abduct'].state.data) if run['Abduct'].state else 0
                print(f"  Files downloaded: {file_count}")
            
            # Show processing info if available
            if 'Processor' in run and run['Processor']:
                print(f"  Processing completed: Yes")
    
    try:
        results = await beam.go(cb=detailed_callback)
        print("\nDetailed tracking execution completed!")
        return results
    except Exception as e:
        print(f"Error: {e}")
        return None

# Run the detailed example (uncomment to execute)
# detailed_results = await run_with_detailed_tracking()
print("Detailed tracking function defined. Uncomment the line above to run.")

## 6. Configuration Management

Learn how to save, load, and manage configuration files.

In [ ]:
# Save configurations to files
config_dir = Path("./tutorial_configs")
config_dir.mkdir(exist_ok=True)

# Save each configuration
configs_to_save = [
    ("simple.json", simple_config),
    ("advanced.json", advanced_config),
    ("monitoring.json", monitoring_config)
]

for filename, config in configs_to_save:
    config_path = config_dir / filename
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    print(f"Saved {filename}")

print(f"\nAll configurations saved to {config_dir}")

In [ ]:
# Load configurations from files
def load_config_from_file(file_path):
    """Load a configuration from a JSON file"""
    try:
        with open(file_path, 'r') as f:
            config = json.load(f)
        print(f"✅ Loaded {file_path.name} successfully")
        return config
    except Exception as e:
        print(f"❌ Failed to load {file_path.name}: {e}")
        return None

# Load all saved configurations
loaded_configs = {}
for filename, _ in configs_to_save:
    config_path = config_dir / filename
    if config_path.exists():
        loaded_config = load_config_from_file(config_path)
        if loaded_config:
            loaded_configs[filename] = loaded_config

print(f"\nLoaded {len(loaded_configs)} configurations")

## 7. Real-World Examples

Here are some practical examples for common use cases.

In [ ]:
# Example: Documentation scraper
documentation_config = {
    "role": "server",
    "settings": {
        "name": "documentation_scraper",
        "proj_dir": "./docs_output",
        "jobs": [
            {
                "url": "https://docs.python.org/3/tutorial/",
                "types": ["html"],
                "beacon": None,
                "delay": None,
                "tasks": ["abduct", "visits", "process"],
                "description": "Scrape Python tutorial documentation",
                "recursive": True,
                "max_depth": 2
            }
        ]
    }
}

print("Documentation scraper configuration created")
print("This would scrape Python documentation with 2 levels of depth")

In [ ]:
# Example: Research paper collector
research_config = {
    "role": "server",
    "settings": {
        "name": "research_papers",
        "proj_dir": "./research_output",
        "jobs": [
            {
                "url": "https://arxiv.org/list/cs.AI/recent",
                "types": ["pdf", "html"],
                "beacon": None,
                "delay": None,
                "tasks": ["abduct", "visits", "process"],
                "description": "Collect recent AI research papers",
                "custom": [
                    {
                        "func": "arxiv_processor",
                        "headers": {
                            "User-Agent": "Academic-Research-Bot/1.0"
                        }
                    }
                ]
            }
        ]
    }
}

print("Research paper collector configuration created")
print("This would collect recent AI papers from arXiv")

In [ ]:
# Example: News monitoring system
news_config = {
    "role": "watcher",
    "settings": {
        "name": "news_monitor",
        "proj_dir": "./news_output",
        "jobs": [
            {
                "url": "https://feeds.bbci.co.uk/news/technology/rss.xml",
                "types": ["xml", "html"],
                "beacon": None,
                "delay": 1800,  # Check every 30 minutes
                "tasks": ["abduct", "visits", "process"],
                "description": "Monitor BBC Technology news",
                "custom": [
                    {
                        "func": "rss_processor",
                        "headers": {
                            "User-Agent": "News-Monitor-Bot/1.0"
                        }
                    }
                ]
            }
        ]
    }
}

print("News monitoring configuration created")
print("This would monitor BBC Technology news every 30 minutes")

## 8. Best Practices and Tips

Here are some important best practices for using tractor-beam effectively.

In [ ]:
# Best practices example
best_practices_config = {
    "role": "server",
    "settings": {
        "name": "best_practices_example",
        "proj_dir": "./best_practices_output",
        "jobs": [
            {
                "url": "https://httpbin.org/robots.txt",
                "types": ["txt"],
                "beacon": None,
                "delay": 1,  # Respectful delay
                "tasks": ["abduct", "visits"],
                "description": "Respectful scraping with proper delays",
                "custom": [
                    {
                        "func": "respectful_scraper",
                        "headers": {
                            "User-Agent": "TractorBeam/1.0 (+https://your-website.com/bot)",
                            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
                            "Accept-Language": "en-US,en;q=0.5",
                            "Accept-Encoding": "gzip, deflate",
                            "DNT": "1",
                            "Connection": "keep-alive",
                            "Upgrade-Insecure-Requests": "1"
                        }
                    }
                ],
                "overwrite": False,  # Don't overwrite existing files
                "max_depth": 3,      # Limit recursion depth
                "output_dir": "./organized_output"  # Organized output
            }
        ]
    }
}

print("Best practices configuration created with:")
print("- Respectful delays")
print("- Proper User-Agent headers")
print("- Limited recursion depth")
print("- Organized output structure")
print("- Non-destructive operations")

## 9. Troubleshooting and Debugging

Common issues and how to debug them.

In [ ]:
# Debugging utilities
def debug_configuration(config):
    """Debug a configuration for common issues"""
    print("🔍 Debugging configuration...")
    
    # Check basic structure
    required_fields = ['role', 'settings']
    for field in required_fields:
        if field not in config:
            print(f"❌ Missing required field: {field}")
        else:
            print(f"✅ Found required field: {field}")
    
    # Check settings
    if 'settings' in config:
        settings = config['settings']
        settings_fields = ['name', 'proj_dir', 'jobs']
        
        for field in settings_fields:
            if field not in settings:
                print(f"❌ Missing settings field: {field}")
            else:
                print(f"✅ Found settings field: {field}")
        
        # Check jobs
        if 'jobs' in settings:
            jobs = settings['jobs']
            print(f"📝 Found {len(jobs)} job(s)")
            
            for i, job in enumerate(jobs):
                print(f"   Job {i+1}: {job.get('url', 'No URL')}")
                if not job.get('url'):
                    print(f"   ⚠️  Job {i+1} missing URL")
                if not job.get('tasks'):
                    print(f"   ⚠️  Job {i+1} missing tasks")
    
    print("🔍 Debugging complete")

# Test debugging function
print("Testing debug function with simple configuration:")
debug_configuration(simple_config)

In [ ]:
# Error handling example
async def safe_execution(config, description=""):
    """Safely execute a configuration with proper error handling"""
    print(f"🚀 Starting safe execution: {description}")
    
    try:
        # Validate configuration first
        config_obj = Config(config)
        print("✅ Configuration validation passed")
        
        # Create beam
        beam = tractor.Beam(config)
        print("✅ Beam created successfully")
        
        # Execute with error handling callback
        async def error_aware_callback(runs):
            print(f"📊 Progress: {len(runs)} runs completed")
            
            # Check for errors in runs
            for i, run in enumerate(runs):
                status = run.get('status', 'unknown')
                if status == 'error':
                    print(f"❌ Run {i+1} encountered an error")
                elif status == 'complete':
                    print(f"✅ Run {i+1} completed successfully")
                else:
                    print(f"ℹ️  Run {i+1} status: {status}")
        
        # Execute
        results = await beam.go(cb=error_aware_callback)
        print("🎉 Execution completed successfully")
        return results
        
    except Exception as e:
        print(f"❌ Execution failed: {type(e).__name__}: {e}")
        
        # Provide helpful error messages
        if "Config" in str(e):
            print("💡 Tip: Check your configuration structure and required fields")
        elif "URL" in str(e) or "HTTP" in str(e):
            print("💡 Tip: Check your internet connection and URL accessibility")
        elif "Permission" in str(e):
            print("💡 Tip: Check file/directory permissions")
        else:
            print("💡 Tip: Enable debug logging for more details")
        
        return None

# Example of safe execution (uncomment to run)
# result = await safe_execution(simple_config, "Simple configuration test")
print("Safe execution function defined. Uncomment the line above to run with error handling.")

## 10. Conclusion

This tutorial has covered:

1. **GUI Interface**: User-friendly graphical interface for all operations
2. **Configuration Management**: Creating, validating, and managing configurations
3. **Programmatic Usage**: Using tractor-beam in code with callbacks and error handling
4. **Real-World Examples**: Practical configurations for common use cases
5. **Best Practices**: Respectful scraping and proper configuration
6. **Debugging**: Tools and techniques for troubleshooting

### Next Steps

1. **Try the GUI**: Run `python examples/gui_demo.py` to explore the interface
2. **Experiment**: Create your own configurations for your specific needs
3. **Explore Beacons**: Look into specialized beacons for different data sources
4. **Scale Up**: Use parallel processing for large-scale data collection

### Resources

- **GUI Quick Start**: `examples/tutorial/gui_quickstart.md`
- **Complete Tutorial**: `examples/tutorial/complete_tutorial.md`
- **Example Configurations**: `examples/configs/`
- **GUI Demo**: `examples/gui_demo.py`

Happy scraping with Tractor Beam! 🛸